# ⚡ Turjo 4K Vision Studio — Free Cloud GPU Notebook
> **Generate Studio-Grade 4K (3840×2160) Photorealistic Images for 100% Free on Google Colab / Kaggle.**

### System Pipeline:
1. **Stage 1**: SDXL RealVisXL V4.0 (Base 16:9 Synthesis @ 1280x720)
2. **Stage 2**: Img2Img Micro-Detail Latent Refiner (Denoise 0.28)
3. **Stage 3**: Real-ESRGAN Neural Super-Resolution (Upscale 3x to 3840x2160)
4. **Stage 4**: HDR Color & Contrast Grading

In [ ]:
# @title Step 1: Install Free Cloud Dependencies
!pip install -q diffusers transformers accelerate safetensors realesrgan basicsr torchvision Pillow opencv-python

In [ ]:
# @title Step 2: Configure Your Prompt
PROMPT = "Ultra-realistic luxury sports car, sleek aerodynamic design, glossy metallic body, dramatic front three-quarter view, parked on a modern city street at night, cinematic lighting, realistic reflections, premium automotive photography, sharp details, realistic materials, shallow depth of field, dramatic atmosphere, professional studio-quality composition, photorealistic, HDR, ultra-detailed, 8k uhd"
NEGATIVE_PROMPT = "low quality, blurry, deformed, cartoon, 3d render, plastic finish, bad proportions, duplicate wheels, oversaturated, text, watermark"

In [ ]:
# @title Step 3: Run Full 4K Progressive Synthesis Pipeline
import gc
import torch
import cv2
import numpy as np
from PIL import Image, ImageEnhance
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Active Device: {device}")

# 1. Base Latent Generation (SDXL RealVisXL V4.0)
print("[Stage 1/4] Loading SDXL RealVisXL (FP16)... ")
pipe = AutoPipelineForText2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    variant="fp16" if device == "cuda" else None,
    use_safetensors=True
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
if device == "cuda":
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_tiling()
else:
    pipe.to(device)

print("Generating Base 1280x720 Render...")
base_image = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=1280,
    height=720,
    guidance_scale=6.5,
    num_inference_steps=32
).images[0]

# 2. Micro-Texture Refiner
print("[Stage 2/4] Refining Surface Reflections (Denoise 0.28)...")
refiner_pipe = AutoPipelineForImage2Image.from_pipe(pipe)
refined_image = refiner_pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=base_image,
    strength=0.28,
    guidance_scale=6.0,
    num_inference_steps=20
).images[0]

# Free memory for Neural Upscaler
del pipe, refiner_pipe
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# 3. Tiled Neural 4K Upscaling
print("[Stage 3/4] Neural Upscaling to 4K (3840x2160)...")
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upscaler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=model,
    tile=512,
    tile_pad=10,
    pre_pad=0,
    half=True if device == "cuda" else False,
    device=device
)

img_cv = cv2.cvtColor(np.array(refined_image), cv2.COLOR_RGB2BGR)
output_cv, _ = upscaler.enhance(img_cv, outscale=3.0)
raw_4k = Image.fromarray(cv2.cvtColor(output_cv, cv2.COLOR_BGR2RGB))

# 4. Color Grading
sharpness = ImageEnhance.Sharpness(raw_4k).enhance(1.12)
final_image = ImageEnhance.Contrast(sharpness).enhance(1.06)
final_image.save("final_car_4k.png", format="PNG", quality=100)

print("\n[SUCCESS] 4K Image Render Complete: final_car_4k.png (3840 x 2160)")
final_image.show()

In [ ]:
# @title Step 4: Download 4K Image to your Local PC
from google.colab import files
files.download('final_car_4k.png')